# Instant gratification

## An agent harness from first principles

In this notebook we build an AI agent that finds its way out of a maze.

Everything here runs on free models. The only setup is an OpenRouter account and an API key.

This should be fun regardless of your level of experience.

## Setup part 1: create your OpenRouter account

OpenRouter is a single service that gives you access to hundreds of models, including free ones. We use it throughout the course.

If you already have an OpenRouter account, sign in and go straight to part 2.

1. Open a new browser tab and go to **https://openrouter.ai**
2. Click the **Sign Up** button at the top right of the page.
3. Create your account with the Google button, the GitHub button or your email address. If you choose email, click the link in the verification email that OpenRouter sends you.
4. You do not need to add a credit card or buy credits for this notebook.

## Setup part 2: allow free models in your privacy settings

Free models will not run until you have done this step.

The providers of free models may train on your prompts or publish them. Nothing we send in this course is private.

1. Make sure you are signed in to OpenRouter, then go to **https://openrouter.ai/settings/privacy**
2. Find the setting called **Enable free endpoints that may train on inputs** and switch it **ON**.
3. Find the setting called **Enable free endpoints that may publish prompts** and switch it **ON**.

## Setup part 3: create your API key

If you already have an OpenRouter key saved from another course, use it and go straight to part 4.

1. Go to **https://openrouter.ai/settings/keys**
2. Click the **Create API Key** button.
3. In the **Name** box, type `fde-course`. Leave everything else as it is.
4. Click **Create**.
5. Your new key appears on the screen. It is a long piece of text that starts with `sk-or-v1-`
6. Copy the whole key using the copy button next to it. **OpenRouter only shows you the key once.** Keep this tab open until you have finished part 4. If you lose the key, delete it on this page and create another one.

Never paste your key into a notebook cell. The next step stores it in Colab secrets instead.

## Setup part 4: add the key to Colab as a secret

1. Come back to this notebook in Colab.
2. Look at the narrow strip of icons down the **left edge** of the Colab window. Click the icon that looks like a **key**. A panel called **Secrets** opens.
3. Click **Add new secret**.
4. In the **Name** box, type this exactly, in capital letters, with underscores and with no spaces:

   `OPENROUTER_API_KEY`

5. Click in the **Value** box and paste your key (Ctrl+V on a PC, Cmd+V on a Mac).
6. Switch **ON** the **Notebook access** toggle on the same row.
7. Close the Secrets panel by clicking the key icon again.

## Now for the code

Run each cell in turn by clicking into it and pressing **Shift + Enter**.

In [1]:
import json
import requests
from bs4 import BeautifulSoup
from google.colab import userdata
from IPython.display import Markdown, display
from openai import OpenAI

### Check your key

This cell reads your key from the Colab secret. If Colab asks whether to grant this notebook access to your secret, click **Grant access**.

If you see **SecretNotFoundError**, check that the secret's name is exactly `OPENROUTER_API_KEY` (Setup part 4).

In [2]:
openrouter_api_key = userdata.get("OPENROUTER_API_KEY").strip()

if openrouter_api_key.startswith("sk-or-v1-"):
    print(f"OpenRouter API key exists and begins {openrouter_api_key[:9]}")
else:
    print("This does not look like an OpenRouter API key. Please repeat Setup parts 3 and 4.")

OpenRouter API key exists and begins sk-or-v1-


### Connect to OpenRouter

We use the OpenAI Python package and point it at OpenRouter by changing the base URL.

The model `openrouter/free` picks one of the available free models for each request. We print the name of the model that answered.

In [3]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openrouter/free"

# Retry up to 5 times when a free model is busy
openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL, timeout=120, max_retries=5)

In [4]:
messages = [{"role": "user", "content": "Hi there!"}]

response = openrouter.chat.completions.create(model=MODEL, messages=messages)

print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by nvidia/nemotron-3.5-super-vl:free


Hi there! How can I help you today?

If you see a reply, everything is set up.

If you see an error that mentions **data policy** or **No endpoints found**, go back to Setup part 2 and check that both switches are on. If you see an error that mentions **401**, repeat Setup parts 3 and 4 to get a new key.

## Part 1 of the harness: the tool

A tool is a Python function that the model can ask us to run.

Our agent has one tool, `read_web_page`. It returns the text of a web page without the HTML. It only reads pages in the maze. For any other address, it tells the model which URLs it can read.

In [ ]:
MAZE_URL = "https://maze.edwarddonner.com"

def read_web_page(url):
    """Read a web page in the maze and return the text on the page, without the HTML"""
    print(f"Reading web page: {url}")
    if not url.startswith(MAZE_URL):
        return f"Sorry, this tool can only read pages in the maze. Please ask for a URL that starts with {MAZE_URL}"
    response = requests.get(url, timeout=20)
    page = BeautifulSoup(response.text, "html.parser")
    return page.body.get_text(separator="\n", strip=True)

In [ ]:
print(read_web_page("https://maze.edwarddonner.com"))

Here is what happens with an address outside the maze.

In [ ]:
print(read_web_page("https://edwarddonner.com"))

The model never sees our Python code. We describe the tool to it in JSON with the tool's name, what it does and the arguments it takes. The model reads this description and decides when to use the tool.

In [ ]:
read_web_page_json = {
    "type": "function",
    "function": {
        "name": "read_web_page",
        "description": "Read a web page and return the text on the page",
        "parameters": {
            "type": "object",
            "properties": {
                "url": {"type": "string", "description": "The full URL of the web page to request"},
            },
            "required": ["url"],
        },
    },
}

tools = [read_web_page_json]

When the model wants to use the tool, it replies with the tool's name and the arguments as JSON. Our code runs the function and sends back the result.

If the tool fails, we send the error message back to the model as the result.

In [ ]:
def handle_tool_call(tool_call):
    """Run the tool that the model asked for and package the result as a message"""
    try:
        arguments = json.loads(tool_call.function.arguments)
        result = read_web_page(**arguments)
    except Exception as error:
        result = f"The tool call failed: {error}"
    return {"role": "tool", "tool_call_id": tool_call.id, "content": result}

## Using the tool once

Now we call the model with the tool and ask it to describe the maze's front page. The model replies with a request to use the tool.

In [ ]:
messages = [{"role": "user", "content": "Please read https://maze.edwarddonner.com and tell me what it says. Just describe the page for now, without going into the maze."}]

response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
message = response.choices[0].message

print(f"Answered by {response.model}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(message.tool_calls)

The finish reason is `tool_calls`. The model has asked for `read_web_page` with the maze's address.

We add the model's request and the tool's result to the conversation and call the model again.

Now and then a small model asks for the tool again instead of answering. If the output is blank, run the two cells again.

In [ ]:
messages.append(message)
for tool_call in message.tool_calls:
    messages.append(handle_tool_call(tool_call))

response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)

print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

To walk the maze, the model needs a tool call for every location it visits.

## Part 2 of the harness: the loop

First, a system prompt that asks the model to reply in Markdown.

In [ ]:
SYSTEM_PROMPT = """
You are an agent that completes tasks by using your tools, one step at a time.
Your replies are displayed as Markdown, so format them in Markdown with headings, bold text and lists.
Put any map or diagram in a code block so that it lines up. Do not use emojis.
"""

The agent is the code from the last section inside a loop. The loop stops when the model replies with text and no tool call, or after 25 steps. If a free model sends back an empty reply, we ask again.

In [ ]:
def run_agent(task, max_steps=25):
    """Call the model in a loop, running the tool whenever it asks, until it gives a final answer"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        message = response.choices[0].message
        print(f"Step {step} answered by {response.model}")

        text = (message.content or "").strip()
        if text:
            display(Markdown(text))

        if message.tool_calls:
            messages.append(message)
            for tool_call in message.tool_calls:
                messages.append(handle_tool_call(tool_call))
        elif text:
            return
        else:
            print("The reply was empty, so we ask again")

    print("The agent ran out of steps")

## Into the maze

I have built a small maze at **https://maze.edwarddonner.com**

Each location is a web page that describes a room and lists its exits. Some passages are dead ends. There is a message waiting at the way out.

We give the agent the maze's address and ask it to do what the page says. A run takes a minute or two. If you see an error, or the agent stops before it finds the way out, run the cell again.

In [ ]:
run_agent("Have a look at https://maze.edwarddonner.com and do what it says.")

## What just happened

The model read each page, picked an exit, backed out of dead ends and carried on until it found the message.

Several different models probably took turns answering. Each one worked from the same conversation history.

The code around the model is called the agent harness. We return to harness engineering at the end of Week 1.

### Try it yourself

- Run the last cell again. Does the agent take the same route?
- Change `MODEL` to one named free model from [free popular models on OpenRouter](https://openrouter.ai/models?max_price=0&order=most-popular) and compare the number of steps it takes.

### Didn't work?

Try again, or you can bring up my results here to see my run instead:

[Ed's completed colab](https://colab.research.google.com/drive/1W40cKufmBoioV3lgXMMj2zb0-RP-ZnD_?usp=sharing)

A free OpenRouter account allows 50 requests a day to free models. A run of the maze uses 10 to 15.

If you got unlucky and your LLM got lost in the maze and used up your quota, then it's no big deal! It's also fun that you were able to see an LLM get stuck solving a problem. Your quota will reset within a day.